# Predicting the motion of a pendulum using neural networks

This lab will build simple neural networks using tensorflow and keras, and them to predict the angular behaviour of two pendulum systems.

## Step 1: Generating the data
We'll use some simple, synthetic data to train the network.   
Let's generate some quickly using an analytic expression. 
$$ \theta (t) = \theta_0 \cos \left(\sqrt{\tfrac{g}{L}} t\right)$$

**(i) Confirm that this expression is correct: what are $\theta_0$, $g$, and $L$?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#analytic function 
def pendulum_motion(theta0, L, g, t):
    return theta0 * np.cos(np.sqrt(g / L) * t)

# parameters
theta0 = 0.3
L = 0.12

# Generate data
t = np.linspace(0, 1,300)  # Time array
theta = pendulum_motion(theta0=theta0, L=L, g=9.81, t=t) 

We're going to try and learn the angle (from the time) for a given set of data

In [ ]:
# plot the data
fig = plt.figure()
plt.plot(t, theta, 'o')
plt.ylabel(r'Angle,  $\theta$ (rad.)', fontsize=14)
plt.xlabel(r'Time, t (s)', fontsize=14)

plt.show()

## Step 2: Preparing the data
As usual, we need separate test, train and validation sets.  
We'll use the tools we've previously seen in scikit-learn for this.


In [ ]:
# import the train_test_split tool from scikit-learn
from sklearn.model_selection import train_test_split


In [ ]:
# split the entire data set into training and split sets
x_train, x_test, y_train, y_test = train_test_split(t, theta, test_size=0.125, random_state=0)

# split the training set into training and validation sets
x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.2, random_state=1)


In [ ]:
print(x_train.size, x_valid.size, x_test.size)

In [ ]:
# plot the data
fig = plt.figure()
plt.plot(x_train, y_train, 'o', label='training data')
plt.plot(x_valid, y_valid, 'o', label='validation data')
plt.plot(x_test, y_test, 'o', label='testing data')

plt.ylabel(r'Angle,  $\theta$ (rad.)', fontsize=14)
plt.xlabel(r'Time, t (s)', fontsize=14)
plt.legend()
plt.show()

## Step 3: Building a neural network

We'll use the "Sequential" model class and the "Dense" layers class from keras to build a simple fully-connected network layer-by-layer.


The key steps are:
- define the model and its geometry (number of layers, neurons, set initial weights and biases, choose activation functions, etc.)
- *compile* the model (specify how it will be optimised, additional metrics to calculate at each epoch)
- *fit* the model using the training data


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
model = Sequential()
model.add(Dense(40, input_dim=1, activation='relu'))  # First hidden layer (with 40 neurons)
                                                      # Input layer is a single neuron (i.e. the time value)
model.add(Dense(20, activation='relu'))              # Second Hidden layer
model.add(Dense(1, activation='linear'))             # Output layer

**(ii) Why does the last layer here have a linear activation function?**   
 (e.g. just returns a weighted sum of inputs and bias, without applying a non-linear transformation)

In [ ]:
# can view a basic summary of the model using...
model.summary()

In [ ]:
# can view the weights and biases of each layer
#e.g. for the first layer
weights, biases = model.layers[0].get_weights()

In [ ]:
# print(weights)
# print(weights.shape)
# print(biases)

The next two cells compile and train the model.   
Note that if you rerun .fit(), _without_ recompiling, it will use the previous optimised weights as a starting point.  
You need to redefine (the model = .... cell above) and then recompile it in order to reset the weights and biases.

In [ ]:
# compile the model
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss='mean_squared_error', metrics=["mae"])

In [ ]:
# Fit the model with the training data 
# outputs the metrics, which we store here to 'history':

history = model.fit(x_train, y_train, epochs=100, batch_size=16, validation_data=[x_valid, y_valid])

The training of the network is usually examined using a *training curve*, which plots the loss on both training and validation sets per epoch.   
**(iii) Why do we plot both?**

In [ ]:
# Plot the evolution of loss as the network learns
figloss=plt.figure()
ax = figloss.add_subplot(111)
ax.plot(history.history['loss'], label='Training')
ax.plot(history.history['val_loss'], label='Validation')
ax.set_ylabel('Loss', fontsize=16)
ax.set_xlabel('Epoch', fontsize=16)
ax.legend(loc='upper right', fontsize=14)
ax.tick_params(axis='both', which='both', labelsize=12)
plt.show()

## Step 4: Make predictions and evaluate model

As with previous examples, we can apply a .predict() method to use our trained model to make predictions on another set of data 

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
# First, let's compare out predictions to the analytic curve
# (in most problems, we won't have this!)
plt.plot(x_test, y_pred, 'o')
plt.plot(t, theta, '--')
plt.ylabel(r'Angle,  $\theta$ (rad.)', fontsize=14)
plt.xlabel(r'Time, t (s)', fontsize=14)
plt.show()


In [ ]:
# A more typical plot (for regression) is to plot the predicted against the true values
# (this is somewhat similar to the confusion matrix in classification!)
# the diagonal line (x=y) indicates perfect agreement: we want our results to fall upon this line.

fig=plt.figure()
ax=fig.add_subplot()
ax.plot(y_test, y_pred, 'o')
ax.plot([-theta0,theta0],[-theta0,theta0], '--') 
ax.set_aspect('equal')
ax.set_xlabel(r'True $\theta$', fontsize=16) 
ax.set_ylabel(r'Predicted $\theta$', fontsize=16) 

plt.show()



In [ ]:
# The .evaluate() method allows you to calculate the loss (and any other metrics specified when compiling the model) on the test set
model.evaluate(x_test, y_test)